# Sprint 4 · Task 6: Experiment Record


| # | Experiment | What changes |
|---|---|---|
| 1 | Change the earlier year | `EARLIER_YEAR`: 2016 → 2022 |
| 2 | Change the date window | End of the imagery window: 31 Dec → 1 Oct |
| 3 | Investigate an unusual result | Count the Landsat images per year (no change to the analysis) |

Every experiment follows the Task 6 pattern: **What I changed → What I expected → What happened → What I learned.**

The notebook passes each change as an **argument**, so the standard GeoRAD settings (2016 vs 2026, threshold 0.0, cloud limit 40) are never overwritten. Your main Sprint 4 dataset is not affected.

> Run the cells from top to bottom. Section 4 takes a few minutes because it asks Earth Engine for each town and year.

## 1. Setup

In [1]:
%pip install -q earthengine-api pandas matplotlib geopandas folium mapclassify

In [2]:
from pathlib import Path
import json
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

try:
    from google.colab import files  # type: ignore
    print("Upload your towns GeoJSON (the same file used in the main Sprint 4 notebook).")
    geo_uploaded = files.upload()
    GEOJSON_PATH = next(iter(geo_uploaded))
except Exception:
    GEOJSON_PATH = "sample_towns.geojson"

gdf = gpd.read_file(GEOJSON_PATH)
print(f"Loaded GeoJSON: {GEOJSON_PATH}")
print(f"Features: {len(gdf)} | CRS: {gdf.crs}")
print(f"Geometry types: {Counter(gdf.geometry.geom_type)}")
print(f"Property columns: {list(gdf.columns)}")

PREFERRED_NAME_FIELDS = ["wardname", "ward", "town", "town_name", "name", "NAME", "Town"]
TOWN_NAME_FIELD = None  # Override if needed, e.g. TOWN_NAME_FIELD = "ward"

if TOWN_NAME_FIELD is None:
    for field in PREFERRED_NAME_FIELDS:
        if (
            field in gdf.columns
            and gdf[field].notna().all()
            and (gdf[field].astype(str).str.strip() != "").all()
        ):
            TOWN_NAME_FIELD = field
            break

if TOWN_NAME_FIELD is None:
    raise ValueError(
        "Could not detect a town-name field. Set TOWN_NAME_FIELD manually. "
        f"Columns: {list(gdf.columns)}"
    )

gdf = gdf.copy()
gdf["Location"] = gdf[TOWN_NAME_FIELD].astype(str).str.strip()

towns_wgs = gdf.to_crs(epsg=4326) if gdf.crs and gdf.crs.to_epsg() != 4326 else gdf.copy()
if towns_wgs.crs is None:
    towns_wgs = towns_wgs.set_crs(epsg=4326)

print(f"Using town-name field: {TOWN_NAME_FIELD}")
print("Locations:", sorted(towns_wgs["Location"].unique().tolist()))

Upload your towns GeoJSON (the same file used in the main Sprint 4 notebook).


Saving Five_Towns.geojson to Five_Towns.geojson
Loaded GeoJSON: Five_Towns.geojson
Features: 5 | CRS: EPSG:4326
Geometry types: Counter({'MultiPolygon': 5})
Property columns: ['globalid', 'uniq_id', 'timestamp', 'editor', 'wardname', 'wardcode', 'lganame', 'lgacode', 'statename', 'statecode', 'amapcode', 'status', 'source', 'urban', 'geometry']
Using town-name field: wardname
Locations: ['Bale', 'Coker', 'Jinadu Aiyetoro', 'Osho', 'Savage']


## 2. Connect to Google Earth Engine

Paste your **Project ID** (not the project name).

In [3]:
import ee

EE_PROJECT = ""  # <-- paste your project ID here

try:
    from google.colab import userdata  # type: ignore
    secret_project = userdata.get("EE_PROJECT")
    if secret_project and not EE_PROJECT:
        EE_PROJECT = secret_project
except Exception:
    pass

if not EE_PROJECT:
    raise ValueError(
        "Set EE_PROJECT to your Google Cloud / Earth Engine project ID "
        "(or add Colab Secret EE_PROJECT)."
    )

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print(f"Earth Engine initialized for project: {EE_PROJECT}")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized for project: georad-sprint4-ee


## 3. Settings and helper functions

The **standard settings** are the GeoRAD defaults and form the baseline.
The helper functions are the same as in the main notebook (cloud mask, yearly median, NDBI, built-up share). The only additions are arguments for the **date window** and **cloud limit**, so the experiments can change them without editing global settings.

In [4]:
# ---- Standard GeoRAD settings (baseline: do not change) ----
EARLIER_YEAR = 2016
RECENT_YEAR = 2026
NDBI_THRESHOLD = 0.0
ANALYSIS_SCALE_M = 30
MAX_CLOUD_COVER = 40
LOW_MAX = 5.0
AVERAGE_MAX = 15.0
STANDARD_END = "12-31"   # standard imagery window: 1 Jan to 31 Dec

# ---- Experiment settings ----
EXP1_EARLIER_YEAR = 2022        # Experiment 1: the alternative earlier year
EXP2_END = "10-01"              # Experiment 2: shorter window (1 Jan to 1 Oct)
EXP3_YEARS = [2016, 2022, 2025, 2026]   # Experiment 3: years to count images for

print(f"Baseline: {EARLIER_YEAR} vs {RECENT_YEAR} | threshold={NDBI_THRESHOLD} | cloud<{MAX_CLOUD_COVER}% | window 01-01 to {STANDARD_END}")

Baseline: 2016 vs 2026 | threshold=0.0 | cloud<40% | window 01-01 to 12-31


In [5]:
L8 = "LANDSAT/LC08/C02/T1_L2"
L9 = "LANDSAT/LC09/C02/T1_L2"


def mask_landsat_sr(image):
    # Cloud/shadow mask + optical scaling for Landsat Collection 2 Level-2
    qa = image.select("QA_PIXEL")
    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)        # dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
    )
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True).updateMask(mask)


def image_pile(path, year, region, end_md, max_cloud):
    return (
        ee.ImageCollection(path)
        .filterBounds(region)
        .filterDate(f"{year}-01-01", f"{year}-{end_md}")
        .filter(ee.Filter.lt("CLOUD_COVER", max_cloud))
    )


def year_composite(year, region, end_md=STANDARD_END, max_cloud=MAX_CLOUD_COVER):
    l8 = image_pile(L8, year, region, end_md, max_cloud).map(mask_landsat_sr)
    l9 = image_pile(L9, year, region, end_md, max_cloud).map(mask_landsat_sr)
    merged = l8.merge(l9)
    count = merged.size()
    return ee.Image(
        ee.Algorithms.If(count.gt(0), merged.median().clip(region), ee.Image.constant(0).clip(region))
    )


def image_count(year, region, end_md=STANDARD_END, max_cloud=MAX_CLOUD_COVER):
    n8 = image_pile(L8, year, region, end_md, max_cloud).size()
    n9 = image_pile(L9, year, region, end_md, max_cloud).size()
    return int(n8.add(n9).getInfo())


def ndbi_image(composite):
    # Landsat 8/9: NIR = SR_B5, SWIR1 = SR_B6
    return composite.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")


def development_percent(year, region, end_md=STANDARD_END):
    # Percent of pixels with NDBI above the threshold inside the region
    built = ndbi_image(year_composite(year, region, end_md)).gt(NDBI_THRESHOLD).rename("built")
    stats = built.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=ANALYSIS_SCALE_M,
        maxPixels=1e9,
        bestEffort=True,
    ).getInfo()
    value = stats.get("built")
    return np.nan if value is None else float(value) * 100.0


def gdf_row_to_ee_geometry(row):
    geojson = json.loads(gpd.GeoSeries([row.geometry], crs=towns_wgs.crs).to_json())
    return ee.Geometry(geojson["features"][0]["geometry"])


def classify_change(change_pp):
    if change_pp is None or (isinstance(change_pp, float) and np.isnan(change_pp)):
        return "Unavailable"
    if change_pp < LOW_MAX:
        return "Low"
    if change_pp < AVERAGE_MAX:
        return "Average"
    return "High"


regions = {row["Location"]: gdf_row_to_ee_geometry(row) for _, row in towns_wgs.iterrows()}
Path("task6_outputs").mkdir(exist_ok=True)
print("Helpers ready for:", list(regions))

Helpers ready for: ['Savage', 'Coker', 'Bale', 'Osho', 'Jinadu Aiyetoro']


## 4. Collect the measurements

This measures each town for the years and windows the three experiments need:

- 2016, 2022 and 2026 with the **standard** window (baseline and Experiment 1)
- 2016 and 2026 with the **shorter** window ending 1 October (Experiment 2)

Errors for one town or year are recorded as blanks and do not stop the run.

In [6]:
jobs = [
    (EARLIER_YEAR, STANDARD_END, "standard"),
    (EXP1_EARLIER_YEAR, STANDARD_END, "standard"),
    (RECENT_YEAR, STANDARD_END, "standard"),
    (EARLIER_YEAR, EXP2_END, "short"),
    (RECENT_YEAR, EXP2_END, "short"),
]

records = []
for loc, region in regions.items():
    for yr, end_md, window in jobs:
        try:
            dev = development_percent(yr, region, end_md)
        except Exception as exc:
            dev = np.nan
            print(f"  ! {loc} {yr} (to {end_md}): {exc}")
        records.append({"Location": loc, "Year": yr, "Window": window, "Development": dev})
    print(f"Done: {loc}")

measure_df = pd.DataFrame(records)
measure_df.to_csv("task6_outputs/task6_measurements.csv", index=False)


def dev_lookup(year, window):
    d = measure_df[(measure_df["Year"] == year) & (measure_df["Window"] == window)]
    return d.set_index("Location")["Development"]

print(f"\nCollected {len(measure_df)} measurements.")

Done: Savage
Done: Coker
Done: Bale
Done: Osho
Done: Jinadu Aiyetoro

Collected 25 measurements.


## 5. Baseline (standard settings)

2016 vs 2026, threshold 0.0, cloud limit 40, full-year window. This should match your main Sprint 4 table.

In [7]:
baseline_df = pd.DataFrame(
    {
        "Earlier Development": dev_lookup(EARLIER_YEAR, "standard"),
        "Recent Development": dev_lookup(RECENT_YEAR, "standard"),
    }
)
baseline_df["Development Change"] = baseline_df["Recent Development"] - baseline_df["Earlier Development"]
baseline_df["Level"] = baseline_df["Development Change"].map(classify_change)
baseline_df = baseline_df.round(2)
baseline_df.to_csv("task6_outputs/baseline_standard_settings.csv")
baseline_df

,Earlier Development,Recent Development,Development Change,Level
Location,,,,
Savage,95.89,78.31,-17.57,Low
Coker,92.87,65.51,-27.35,Low
Bale,94.67,87.94,-6.74,Low
Osho,96.48,87.25,-9.23,Low
Jinadu Aiyetoro,89.85,86.72,-3.13,Low


## 6. Experiment 1: Change the earlier year

**What I changed:** `EARLIER_YEAR` from 2016 to 2022.

**What I expected:** The negative change would shrink or disappear if 2016 was an unusual baseline year. *(Edit to match what you actually thought.)*

In [8]:
exp1_df = pd.DataFrame(
    {
        f"{EARLIER_YEAR}": dev_lookup(EARLIER_YEAR, "standard"),
        f"{EXP1_EARLIER_YEAR}": dev_lookup(EXP1_EARLIER_YEAR, "standard"),
        f"{RECENT_YEAR}": dev_lookup(RECENT_YEAR, "standard"),
    }
)
exp1_df[f"Change ({EARLIER_YEAR} to {RECENT_YEAR})"] = exp1_df[f"{RECENT_YEAR}"] - exp1_df[f"{EARLIER_YEAR}"]
exp1_df[f"Change ({EXP1_EARLIER_YEAR} to {RECENT_YEAR})"] = exp1_df[f"{RECENT_YEAR}"] - exp1_df[f"{EXP1_EARLIER_YEAR}"]
exp1_df = exp1_df.round(2)
exp1_df.to_csv("task6_outputs/exp1_earlier_year.csv")

print("Built-up share (%) and change (percentage points):")
display(exp1_df)

gap_base = (exp1_df[f"{EXP1_EARLIER_YEAR}"] - exp1_df[f"{EARLIER_YEAR}"]).round(2)
print(f"\n{EXP1_EARLIER_YEAR} minus {EARLIER_YEAR} (how far the baseline moved, in points):")
print(gap_base.to_string())
print(f"\nChange from {EXP1_EARLIER_YEAR} is still negative for every town:",
      bool((exp1_df[f"Change ({EXP1_EARLIER_YEAR} to {RECENT_YEAR})"] < 0).all()))

Built-up share (%) and change (percentage points):


,2016,2022,2026,Change (2016 to 2026),Change (2022 to 2026)
Location,,,,,
Savage,95.89,91.44,78.31,-17.57,-13.13
Coker,92.87,94.02,65.51,-27.35,-28.51
Bale,94.67,93.51,87.94,-6.74,-5.57
Osho,96.48,96.21,87.25,-9.23,-8.97
Jinadu Aiyetoro,89.85,89.73,86.72,-3.13,-3.01



2022 minus 2016 (how far the baseline moved, in points):
Location
Savage            -4.45
Coker              1.15
Bale              -1.16
Osho              -0.27
Jinadu Aiyetoro   -0.12

Change from 2022 is still negative for every town: True


**What happened:** The 2022 values were close to the 2016 values, and the 2026 values did not move at all, so every change stayed negative.

**What I learned:** The baseline year is not the cause. 2016 and 2022 agree, so the low values are specific to 2026.

## 7. Experiment 2: Change the date window

**What I changed:** The end of the imagery window, from 31 December to 1 October, so 2016 and 2026 cover the same months.

**What I expected:** 2026 was low because it is an incomplete year, so matching the windows would fix it. *(Edit to match what you actually thought.)*

In [9]:
exp2_df = pd.DataFrame(
    {
        f"{EARLIER_YEAR} (full year)": dev_lookup(EARLIER_YEAR, "standard"),
        f"{EARLIER_YEAR} (to {EXP2_END})": dev_lookup(EARLIER_YEAR, "short"),
        f"{RECENT_YEAR} (full year)": dev_lookup(RECENT_YEAR, "standard"),
        f"{RECENT_YEAR} (to {EXP2_END})": dev_lookup(RECENT_YEAR, "short"),
    }
)
exp2_df[f"Change (to {EXP2_END})"] = exp2_df[f"{RECENT_YEAR} (to {EXP2_END})"] - exp2_df[f"{EARLIER_YEAR} (to {EXP2_END})"]
exp2_df = exp2_df.round(2)
exp2_df.to_csv("task6_outputs/exp2_date_window.csv")

print("Built-up share (%) with the full-year window and the shorter window:")
display(exp2_df)

shift_recent = (exp2_df[f"{RECENT_YEAR} (to {EXP2_END})"] - exp2_df[f"{RECENT_YEAR} (full year)"]).abs().max()
shift_earlier = (exp2_df[f"{EARLIER_YEAR} (to {EXP2_END})"] - exp2_df[f"{EARLIER_YEAR} (full year)"]).abs().max()
print(f"\nLargest shift for {RECENT_YEAR} when the window is shortened: {shift_recent:.2f} points")
print(f"Largest shift for {EARLIER_YEAR} when the window is shortened: {shift_earlier:.2f} points")
print(f"Change stays negative for every town: {bool((exp2_df[f'Change (to {EXP2_END})'] < 0).all())}")

Built-up share (%) with the full-year window and the shorter window:


,2016 (full year),2016 (to 10-01),2026 (full year),2026 (to 10-01),Change (to 10-01)
Location,,,,,
Savage,95.89,97.69,78.31,78.31,-19.38
Coker,92.87,91.82,65.51,65.51,-26.31
Bale,94.67,94.67,87.94,87.94,-6.74
Osho,96.48,96.89,87.25,87.25,-9.64
Jinadu Aiyetoro,89.85,90.83,86.72,86.72,-4.10



Largest shift for 2026 when the window is shortened: 0.00 points
Largest shift for 2016 when the window is shortened: 1.80 points
Change stays negative for every town: True


**What happened:** The 2026 values stayed exactly the same, because the 2026 data already ended around early October. The 2016 values moved slightly. All five changes stayed negative.

**What I learned:** The incomplete-year explanation was wrong. The window choice moves the numbers a little but does not reverse the result.

## 8. Experiment 3: Investigate an unusual result (image counts)

**What I changed:** Nothing in the analysis. I counted the Landsat images that pass the 40% cloud filter for each year.

**What I expected:** 2026 would have far fewer images than the other years, giving a weak composite. *(Edit to match what you actually thought.)*

These counts are made **before** pixel-level cloud masking, so they show how many scenes are available, not how much of a small ward is actually clear.

In [10]:
rows = []
for loc, region in regions.items():
    for yr in EXP3_YEARS:
        for end_md in (EXP2_END, STANDARD_END):
            try:
                n = image_count(yr, region, end_md)
            except Exception as exc:
                n = np.nan
                print(f"  ! {loc} {yr} (to {end_md}): {exc}")
            rows.append({"Location": loc, "Year": yr, "Window end": end_md, "Images": n})

exp3_df = pd.DataFrame(rows)
exp3_df.to_csv("task6_outputs/exp3_image_counts.csv", index=False)

print("Images per town and year (cloud limit 40%):")
display(exp3_df.pivot_table(index="Location", columns=["Window end", "Year"], values="Images"))

summary = exp3_df.groupby(["Window end", "Year"])["Images"].agg(["min", "max"])
print("\nSummary across towns:")
display(summary)
print("Counts identical for every town:", bool((summary["min"] == summary["max"]).all()))

Images per town and year (cloud limit 40%):


Window end      10-01                   12-31                  
Year             2016  2022  2025  2026  2016  2022  2025  2026
Location                                                       
Bale              9.0  19.0  13.0  12.0  15.0  29.0  20.0  12.0
Coker             9.0  19.0  13.0  12.0  15.0  29.0  20.0  12.0
Jinadu Aiyetoro   9.0  19.0  13.0  12.0  15.0  29.0  20.0  12.0
Osho              9.0  19.0  13.0  12.0  15.0  29.0  20.0  12.0
Savage            9.0  19.0  13.0  12.0  15.0  29.0  20.0  12.0


Summary across towns:


min  max
Window end Year          
10-01      2016    9    9
           2022   19   19
           2025   13   13
           2026   12   12
12-31      2016   15   15
           2022   29   29
           2025   20   20
           2026   12   12

Counts identical for every town: True


**What happened:** The counts were 9 for 2016, 19 for 2022, 13 for 2025 and 12 for 2026 (window to 1 October), identical for all five towns.

**What I learned:** 2026 has more images than 2016, so the number of images does not explain the drop. The counts are the same for every town because all five sit inside the same Landsat scene. The count is made before pixel-level cloud masking, so it cannot show how much of each small ward is usable.

## 9. Check against the recorded results

This compares your live numbers with the values recorded earlier. Small differences are normal if Earth Engine has added or reprocessed images since. Large differences mean a setting or your GeoJSON differs from the earlier run.

In [11]:
recorded = {
    ("2016", "standard"):  {"Savage": 95.89, "Coker": 92.87, "Bale": 94.67, "Osho": 96.48, "Jinadu Aiyetoro": 89.85},
    ("2022", "standard"):  {"Savage": 91.44, "Coker": 94.02, "Bale": 93.51, "Osho": 96.21, "Jinadu Aiyetoro": 89.73},
    ("2026", "standard"):  {"Savage": 78.31, "Coker": 65.51, "Bale": 87.94, "Osho": 87.25, "Jinadu Aiyetoro": 86.72},
    ("2016", "short"):     {"Savage": 97.69, "Coker": 91.82, "Bale": 94.67, "Osho": 96.89, "Jinadu Aiyetoro": 90.83},
}

rows = []
for (yr, window), vals in recorded.items():
    live = dev_lookup(int(yr), window)
    for town, rec in vals.items():
        if town in live.index:
            rows.append({"Year": yr, "Window": window, "Town": town,
                         "Recorded": rec, "Live": round(live[town], 2),
                         "Difference": round(live[town] - rec, 2)})
        else:
            rows.append({"Year": yr, "Window": window, "Town": town,
                         "Recorded": rec, "Live": None, "Difference": None})

check_df = pd.DataFrame(rows)
display(check_df)
largest = check_df["Difference"].abs().max()
print(f"Largest difference from the recorded values: {largest:.2f} points")

,Year,Window,Town,Recorded,Live,Difference
0,2016,standard,Savage,95.89,95.89,-0.0
1,2016,standard,Coker,92.87,92.87,-0.0
2,2016,standard,Bale,94.67,94.67,0.0
3,2016,standard,Osho,96.48,96.48,-0.0
4,2016,standard,Jinadu Aiyetoro,89.85,89.85,-0.0
5,2022,standard,Savage,91.44,91.44,0.0
6,2022,standard,Coker,94.02,94.02,0.0
7,2022,standard,Bale,93.51,93.51,-0.0
8,2022,standard,Osho,96.21,96.21,0.0
9,2022,standard,Jinadu Aiyetoro,89.73,89.73,0.0


Largest difference from the recorded values: 0.00 points


## 10. Task 4 note: the Development Change level

The standard rule labels a change below 5 percentage points as **Low**. A negative change is below 5, so every town gets **Low** with these results.

In [12]:
level_df = baseline_df[["Development Change", "Level"]].copy()
level_df.columns = ["Development Change (pp)", "Development Change Level"]
level_df

,Development Change (pp),Development Change Level
Location,,
Savage,-17.57,Low
Coker,-27.35,Low
Bale,-6.74,Low
Osho,-9.23,Low
Jinadu Aiyetoro,-3.13,Low


## 11. Optional Experiment 4: Threshold sensitivity (Sprint 4, Section 11, idea 1)

Not required for Task 6, but it shows how much the cut-off matters. Pick one town and test a few thresholds for both years.

In [13]:
DEMO_LOCATION = towns_wgs["Location"].iloc[0]   # change to any Location name
DEMO_THRESHOLDS = [-0.05, 0.0, 0.05, 0.10, 0.15]

region = regions[DEMO_LOCATION]
rows = []
for yr in (EARLIER_YEAR, RECENT_YEAR):
    ndbi = ndbi_image(year_composite(yr, region))
    stack = ee.Image.cat([ndbi.gt(t).rename(f"t{i}") for i, t in enumerate(DEMO_THRESHOLDS)])
    stats = stack.reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                               scale=ANALYSIS_SCALE_M, maxPixels=1e9, bestEffort=True).getInfo()
    for i, t in enumerate(DEMO_THRESHOLDS):
        v = stats.get(f"t{i}")
        rows.append({"Year": yr, "Threshold": t, "Development": np.nan if v is None else v * 100})

demo = pd.DataFrame(rows).pivot(index="Threshold", columns="Year", values="Development")
demo["Change"] = demo[RECENT_YEAR] - demo[EARLIER_YEAR]
print(f"Threshold sensitivity for {DEMO_LOCATION}:")
demo.round(2)

Threshold sensitivity for Savage:


Year,2016,2026,Change
Threshold,,,
-0.05,99.23,91.51,-7.72
0.00,95.89,78.31,-17.57
0.05,78.15,55.72,-22.43
0.10,19.74,29.35,9.62
0.15,0.14,5.37,5.24


**What I changed:** `NDBI_THRESHOLD` across ______.

**What I expected:** ______

**What happened:** ______

**What I learned:** ______

---

## 12. Overall finding for Task 6

The negative Development Change appeared in all five towns and survived three input changes. I could not trace it to the baseline year, the date window or the number of images. It is an **unresolved result that needs further investigation**, and Task 5 asks you not to force a conclusion.

**Points to state in your write-up**

1. **The Low label comes from the rule, not from a finding.** Every town is labelled Low because the changes are negative, and the negative values themselves are unexplained, so the label should be read with caution.
2. **NDBI is a proxy for built-up surfaces, not a count of buildings.** Bare soil and dry ground can also score high. The 2016 values of about 90 to 97% suggest the threshold may be counting more than buildings.
3. **Development Change describes what was observed between the two years. It is not a prediction of future development.**

In [14]:
# Save and download everything from this notebook
import shutil
shutil.make_archive("task6_outputs", "zip", "task6_outputs")
print("Saved to task6_outputs/ and task6_outputs.zip")

try:
    from google.colab import files  # type: ignore
    files.download("task6_outputs.zip")
except Exception:
    print("Not in Colab (or download skipped). Files are in task6_outputs/.")

Saved to task6_outputs/ and task6_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>